<a href="https://colab.research.google.com/github/metarun/Mechanistic-Interpretability-Exploration/blob/main/MLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch.nn as nn
import torch

In [57]:
#residual [6 × 8]
torch.manual_seed(1)
residual = torch.rand(6, 8)
residual

# Define layers
d_model = 8
torch.manual_seed(1)
layer_up = nn.Linear(d_model, 4 * d_model)
gelu = nn.GELU()
layer_down = nn.Linear(4 * d_model, d_model)

hidden = layer_up(residual) # ==> [6*8]@[8 * 32] + B ==> [6 * 32]
hidden

hidden = gelu(hidden) # [6 * 32]
hidden # [6 * 32]

mlp_output = layer_down(hidden) # ==> [6*32]@[32 * 8] + B ==> [6 * 8]
mlp_output

residual_transformed = residual + mlp_output
residual, residual_transformed

(tensor([[0.7576, 0.2793, 0.4031, 0.7347, 0.0293, 0.7999, 0.3971, 0.7544],
         [0.5695, 0.4388, 0.6387, 0.5247, 0.6826, 0.3051, 0.4635, 0.4550],
         [0.5725, 0.4980, 0.9371, 0.6556, 0.3138, 0.1980, 0.4162, 0.2843],
         [0.3398, 0.5239, 0.7981, 0.7718, 0.0112, 0.8100, 0.6397, 0.9743],
         [0.8300, 0.0444, 0.0246, 0.2588, 0.9391, 0.4167, 0.7140, 0.2676],
         [0.9906, 0.2885, 0.8750, 0.5059, 0.2366, 0.7570, 0.2346, 0.6471]]),
 tensor([[ 0.6214,  0.1696,  0.2846,  0.9403,  0.0812,  1.0897,  0.5272,  0.4110],
         [ 0.4564,  0.4887,  0.5360,  0.8263,  0.7252,  0.5181,  0.6362,  0.2629],
         [ 0.4661,  0.5200,  0.8221,  0.9663,  0.3641,  0.4097,  0.5543,  0.1041],
         [ 0.2037,  0.4216,  0.7278,  0.9876,  0.0164,  1.0700,  0.7754,  0.5725],
         [ 0.7671,  0.1099, -0.0088,  0.5395,  0.9332,  0.7592,  0.9599,  0.0451],
         [ 0.8306,  0.2163,  0.7219,  0.7869,  0.3691,  1.0223,  0.3525,  0.3447]],
        grad_fn=<AddBackward0>))

In [59]:
class MLPBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        torch.manual_seed(1)
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        # Standard residual connection: x + MLP(x)
        return x + self.net(x)

# Configuration
torch.manual_seed(1)
d_model = 8
residual = torch.rand(6, d_model)

# Initialize and Run
mlp = MLPBlock(d_model)
residual_transformed = mlp(residual)

print(f"Input shape: {residual.shape}")
print(f"Output shape: {residual_transformed.shape}")
display(residual)
display(residual_transformed)

Input shape: torch.Size([6, 8])
Output shape: torch.Size([6, 8])


tensor([[0.7576, 0.2793, 0.4031, 0.7347, 0.0293, 0.7999, 0.3971, 0.7544],
        [0.5695, 0.4388, 0.6387, 0.5247, 0.6826, 0.3051, 0.4635, 0.4550],
        [0.5725, 0.4980, 0.9371, 0.6556, 0.3138, 0.1980, 0.4162, 0.2843],
        [0.3398, 0.5239, 0.7981, 0.7718, 0.0112, 0.8100, 0.6397, 0.9743],
        [0.8300, 0.0444, 0.0246, 0.2588, 0.9391, 0.4167, 0.7140, 0.2676],
        [0.9906, 0.2885, 0.8750, 0.5059, 0.2366, 0.7570, 0.2346, 0.6471]])

tensor([[ 0.6214,  0.1696,  0.2846,  0.9403,  0.0812,  1.0897,  0.5272,  0.4110],
        [ 0.4564,  0.4887,  0.5360,  0.8263,  0.7252,  0.5181,  0.6362,  0.2629],
        [ 0.4661,  0.5200,  0.8221,  0.9663,  0.3641,  0.4097,  0.5543,  0.1041],
        [ 0.2037,  0.4216,  0.7278,  0.9876,  0.0164,  1.0700,  0.7754,  0.5725],
        [ 0.7671,  0.1099, -0.0088,  0.5395,  0.9332,  0.7592,  0.9599,  0.0451],
        [ 0.8306,  0.2163,  0.7219,  0.7869,  0.3691,  1.0223,  0.3525,  0.3447]],
       grad_fn=<AddBackward0>)